**Read csv**

In [0]:
df = spark.read.table("bronze.default.big_mart_sales")

In [0]:
df.show()
display(df)

ingest json

In [0]:
df_json  = spark.read.table("bronze.default.drivers")

In [0]:
df.printSchema()

In [0]:
sales_schema  = '''
Item_Identifier string,
Item_Weight double,
Item_Fat_Content string,
Item_Visibility double,
Item_Type string,
Item_MRP double,
Outlet_Identifier string,
Outlet_Establishment_Year long,
Outlet_Size string,
Outlet_Location_Type string,
Outlet_Type string,
Item_Outlet_Sales double
'''

In [0]:
df_schema = spark.read.schema(sales_schema)

In [0]:
display(df_schema)

In [0]:
df_selected = df.select("Item_Fat_Content","Item_Identifier","Item_Outlet_Sales","Item_Weight","Item_MRP")
display(df_selected)


In [0]:
from pyspark.sql.functions import col , lit

In [0]:
df_selected.filter((col("Item_Outlet_Sales").isNull()))

In [0]:
df_new = df_selected.withColumn("flag",lit("new"))

In [0]:
df_new = df_selected.withColumn("Multiply",col("Item_Weight")*col("Item_MRP"))


In [0]:
display(df_new)

In [0]:
from pyspark.sql.functions import regexp_replace

In [0]:
df_trans = df_new.withColumn("Item_Fat_Content",regexp_replace(col("Item_Fat_Content"),"Low Fat","LF"))\
    .withColumn("Item_Fat_Content",regexp_replace(col("Item_Fat_Content"),"Regular","Reg"))

In [0]:
display(df_trans)

In [0]:
df_cast = df_trans.withColumn("Item_Weight",col("Item_Weight").cast("String"))

In [0]:
from pyspark.sql.functions import col, asc, desc

In [0]:
df_sort = df.sort(col("Item_Weight").desc())
display(df_sort)


In [0]:
df_sort = df_cast.sort("Item_Weight","Item_MRP", ascending=[True, False]) .display()

In [0]:
from pyspark.sql.functions import limit

In [0]:
df_sort.limit(10).display()

In [0]:
from pyspark.sql.functions import dropDuplicates


In [0]:
df_drop = df_sort.dropDuplicates(col("Item_Type")).display()


In [0]:
df_new.dropDuplicates().display()

In [0]:
df_trans.dropDuplicates(subset=["Item_Weight"]).display()

In [0]:
df_cast.distinct().display()

In [0]:
from pyspark.sql.functions import initcap , date_sub
df_cast.select(initcap("Item_Fat_Content")).display()


In [0]:
df_cast.withColumn("Week_Before",date_sub("current_date",7)).display()



In [0]:
from pyspark.sql.functions import current_date  , date_format

In [0]:
df_cast.withColumn("Week_Format", date_format(current_date(), "MM-dd-yyyy")).display()



In [0]:
df.display()

In [0]:
from pyspark.sql.functions import dropna

In [0]:
df.dropna('all').display()

In [0]:
df.dropna('any').display()

In [0]:
df.dropna(subset=['Item_Weight']).display()

In [0]:
df.fillna("Not available").display()

In [0]:
df.fillna(value="Not Available", subset=['Item_Weight']).display()

In [0]:
from pyspark.sql.functions import split

In [0]:
df.withColumn('Outlet_Trans',split('Outlet_Type',' ')).display()

In [0]:
df.withColumn('Outlet_Trans_index',split('Outlet_Type',' ')[1]).display()

In [0]:
from pyspark.sql.functions import explode

In [0]:
df.withColumn('Outlet_Trans', explode('Outlet_Trans')).display()


In [0]:
df.display()

In [0]:
from pyspark.sql.functions import sum

In [0]:
df.groupby('Outlet_Type').agg(sum("Item_MRP")).display()

In [0]:
from pyspark.sql.functions import sum, avg
df.groupby("Item_Type","Outlet_Size").agg(sum("Item_MRP").alias("Total_MRP"), avg("Item_MRP").alias("Avg_MRP")).display()


In [0]:
data = [('User1','Book1'),('User2','Book2'),('User3','Book3'),('User2','Book4')]
schema = 'User string ,book string'
df_book = spark.createDataFrame(data,schema)
df_book.display()                                                             

In [0]:
from pyspark.sql.functions import collect_list
df_book.groupby("User").agg(collect_list("book")).display()


In [0]:
df.groupby("Item_Type").pivot("Outlet_Size").agg(avg("Item_MRP")).display()

In [0]:
from pyspark.sql.functions import when, col

In [0]:
df_when = df.withColumn("Veg_Flag",when(col("Item_Type")=='Meat','Non-Veg').otherwise('Veg')).display()

In [0]:
df_when.withColumn("veg_exp", when((col("Veg_Flag") == "Veg") & (col("Item_MRP") > 100), "Veg_expensive")\
    .when((col("veg_flag") == "Veg") & (col("Item_MRP") < 100), "Veg_inexpensive")\
    .otherwise("Non-Veg"))\
    .display()


In [0]:
dataj1 = [('1','gaur','d01'),
          ('2','kit','d02'),
          ('3','sam','d03'),
          ('4','tim','d03'),
          ('5','aman','d05'),
          ('6','nad','d06')] 

schemaj1 = 'emp_id STRING, emp_name STRING, dept_id STRING' 

df1 = spark.createDataFrame(dataj1,schemaj1)

dataj2 = [('d01','HR'),
          ('d02','Marketing'),
          ('d03','Accounts'),
          ('d04','IT'),
          ('d05','Finance')]

schemaj2 = 'dept_id STRING, department STRING'

df2 = spark.createDataFrame(dataj2,schemaj2)
     


In [0]:
df1.display()

In [0]:
df1 . join(df2,df1.dept_id == df2.dept_id,"inner").display()

In [0]:
df1.join(df2, df1.dept_id == df2.dept_id, "left").display()

In [0]:
df1.join(df2, df1.dept_id == df2.dept_id, "right").display()

In [0]:
df1.join(df2, df1.dept_id == df2.dept_id, "anti").display()